# Setup

In [1]:
!pip install -q datasets transformers evaluate accelerate
!pip install -q "ray[tune]" scipy sklearn torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 8.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating 

In [2]:
!pip -q install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.5/841.5 kB 27.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.2/802.2 kB 32.2 MB/s eta 0:00:00


In [3]:
import os, sys
import pickle
from time import gmtime, strftime
from tqdm.notebook import tqdm  # Progress bar

import matplotlib as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy as sp

import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.utilities import CombinedLoader
from sklearn.metrics import f1_score, classification_report

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, AutoModel, AutoConfig
from transformers import DebertaV2Config, DebertaV2Model, DebertaV2PreTrainedModel
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import get_scheduler


In [4]:
'''Set-up GPU for use'''

gpu_avail = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_avail}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device", device)

# GPU operations have a separate seed
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Some operations on a GPU are implemented stochastic for efficiency
# Ensure that all operations are deterministic on GPU for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Is the GPU available? True
Device cuda


# Prepare Datasets

In [5]:
#ucc_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_no_scores.csv'
ucc_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_no_scores.csv'
ucc_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_confident_no_scores.csv'
#ucc_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_confident_no_sarcasm.csv'

reddit_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_train_balanced.csv'
reddit_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_test_balanced.csv'

In [6]:
# Get max length of text in UCC dataset
df = pd.read_csv(ucc_train_filename)
text_column = df['comment']
max_length_ucc = np.max(np.array([len(comment) for comment in text_column]))
print('UCC max length: ', max_length_ucc)

# Get max length of text in reddit dataset
df = pd.read_csv(reddit_train_filename)
text_column = df['text']
max_length_reddit = np.max(np.array([len(text) for text in text_column]))
print('Reddit max length: ', max_length_reddit)

UCC max length:  349
Reddit max length:  36239


In [7]:
less_100 = 0
less_1000 = 0
less_2000 = 0
less_5000 = 0
less_10000 = 0
less_20000 = 0
greater_20000 = 0
for i in df.index:
    if len(df['text'][i]) < 100:
        less_100 += 1
    elif len(df['text'][i]) < 1000:
        less_1000 += 1
    elif len(df['text'][i]) < 2000:
        less_2000 += 1
    elif len(df['text'][i]) < 5000:
        less_5000 += 1
    elif len(df['text'][i]) < 10000:
        less_10000 += 1
    elif len(df['text'][i]) < 20000:
        less_20000 += 1
    else:
        greater_20000 += 1

print(less_100)
print(less_1000)
print(less_2000)
print(less_5000)
print(less_10000)
print(less_20000)
print(greater_20000)

32
613
498
805
269
57
7


In [8]:
max_length = 1000

# Tokenize data

In [9]:
def preprocess_ucc(example):
    '''Method for preprocessing the UCC dataset'''

    # Gather list of positive labels
    all_labels = []
    for class_ in classes:
        if example[class_] == 1:
            all_labels.append(class_)

    # Convert labels to a vector of binary values
    labels = [0.0 for i in range(len(classes))]
    for label in all_labels:
        label_id = class2id[label]
        labels[label_id] = 1

    example = tokenizer(example['comment'], padding='max_length', truncation=True)
    example['labels'] = labels

    return example

def preprocess_reddit(example):

    if example['labels'] == 'abuse':
        example['labels'] = 1.0
    elif example['labels'] == 'non_abuse':
        example['labels'] = 0.0
    else:
        sys.error(1)

    example = tokenizer(example['text'], padding='max_length', truncation=True)
    return example


ucc = load_dataset('csv', data_files={'train': ucc_train_filename, 'test': ucc_test_filename})
reddit = load_dataset('csv', data_files={'train': reddit_train_filename, 'test': reddit_test_filename})

# Dicts for the UCC dataset to convert between class name and class numerical id
classes = [class_ for class_ in list(ucc['train'].features)[1:] if class_]
class2id = {class_:id for id, class_ in enumerate(classes)}
id2class = {id:class_ for class_, id in class2id.items()}
# Dicts for the Reddit dataset to convert between class name and binary label
class2binary = {'abuse': 1, 'non_abuse': 0}
binary2class = {1:'abuse', 0:'non_abuse'}

# Tokenize Data
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small', model_max_length=max_length, use_fast=False)
tokenized_ucc = ucc.map(preprocess_ucc)
tokenized_reddit = reddit.map(preprocess_reddit)

# Delete class columns from reddit dataset now that we represent the labels as vecs
tokenized_ucc = tokenized_ucc.remove_columns(classes)


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/882 [00:00<?, ? examples/s]

Map:   0%|          | 0/4425 [00:00<?, ? examples/s]

Map:   0%|          | 0/2281 [00:00<?, ? examples/s]

Map:   0%|          | 0/410 [00:00<?, ? examples/s]

In [18]:
#ucc_train_dataset[2]
print(tokenized_ucc, '\n')
print(tokenized_reddit)
print(classes)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 882
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4425
    })
}) 

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2281
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 410
    })
})
['antagonize', 'condescending', 'dismissive', 'hostile', 'sarcastic']


### Finetuning model in native pytorch:

In [11]:
# To prepare the tokenized datasets for training in native pytorch, do the following:

# The model does not accept raw text as an input. Remove the 'comment' and 'text' columns:
tokenized_ucc = tokenized_ucc.remove_columns(['comment'])
tokenized_reddit = tokenized_reddit.remove_columns(['text'])

# Set the format of the dataset to return PyTorch tensors instead of lists:
tokenized_ucc.set_format('torch')
tokenized_reddit.set_format('torch')

# Split the tokenized data into train, test
ucc_train_dataset = tokenized_ucc['train'].shuffle(seed=6)
ucc_eval_dataset = tokenized_ucc['test'].shuffle(seed=6)
reddit_train_dataset = tokenized_reddit['train'].shuffle(seed=6)
reddit_eval_dataset = tokenized_reddit['test'].shuffle(seed=6)

# Note: pytorch requires the label column to be named 'labels'. Change if needed.

# Experiments

## MTL Architecture: Shared base transformer model and separate task heads

## Initialize LLM

In [37]:
#ucc_stl_transformer = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small',
#                                                                 num_labels=len(classes),
#                                                                 id2label=id2class, label2id=class2id,
#                                                                 problem_type='multi_label_classification')
#deberta_config = DebertaV2Config(dropout=0.2, attention_dropout=0.2)

# Set up the LLM
llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
llm = llm.to(device)

In [38]:
# Examine output of LLM by running a test batch through
sample = reddit_eval_dataset[0:2]
inputs = {k: v.to(device) for k, v in sample.items() if k != 'labels'}
outputs = llm(**inputs)

print('Object type: ', type(outputs))
print('Output format (shape): ',outputs[0].shape)      # shape is (N,M,S) where N is num samples, M is num words, and S is size of output vec
print('Output used as input for the classifier (shape): ', outputs[0][:,0,:].shape)

Object type:  <class 'transformers.modeling_outputs.BaseModelOutput'>
Output format (shape):  torch.Size([2, 1000, 768])
Output used as input for the classifier (shape):  torch.Size([2, 768])


## Initialize MTL model and train!

In [12]:
class MTLTextClassification(nn.Module):
    def __init__(self, llm, activation=nn.LeakyReLU()):
        super(MTLTextClassification, self).__init__()

        self.llm = llm
        self.activation = activation

        self.feature_extractor = torch.nn.Sequential(
            nn.Dropout(p=0.2),
            torch.nn.Linear(768, 64),
            torch.nn.LeakyReLU(),
        )

        self.reddit_out = torch.nn.Linear(64, 1)
        self.ucc_out = torch.nn.Linear(64, 5)

    def forward(self, input_ids, token_type_ids, attention_mask, taskid):
        # Get embedding from the LLM we are tuning. Use last hidden state of embedding for classification tasks.
        x = self.llm(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        x = x["last_hidden_state"][:,0,:]

        # Extract shared features for both tasks
        features = self.feature_extractor(x)

        # Branch on task
        if taskid == 1:
            logits = self.ucc_out(features)     # Get output for task 1: UCC classification (BCEWithLogitsLoss)
        elif taskid == 2:
            logits = self.reddit_out(features)  # Get output for task 2: Abuse/Non-abuse classification (BCEWithLogitsLoss)

        return logits


In [40]:
# Create a combined data loaders for both tasks using pytorch-lightning
train_loaders = {
    'a': DataLoader(ucc_train_dataset, shuffle=True, batch_size=4),
    'b': DataLoader(reddit_train_dataset, shuffle=True, batch_size=4),
}
test_loaders = {
    'a': DataLoader(ucc_eval_dataset, shuffle=True, batch_size=4),
    'b': DataLoader(reddit_eval_dataset, shuffle=True, batch_size=4),
}

train_combined_loader = CombinedLoader(train_loaders, 'max_size_cycle')

# Define params of LLM as trainable
for param in llm.parameters():
    param.requires_grad = True

# Initialize Custom MTL Model and optimizer
mtl_model = MTLTextClassification(llm=llm)
mtl_model = mtl_model.to(device)
optimizer = torch.optim.AdamW(mtl_model.parameters(), lr=5e-5)

# Initialize Loss methods for both tasks
ucc_loss = nn.BCEWithLogitsLoss()
reddit_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [32]:
# Examine number of parameters
total_params = sum(p.numel() for p in mtl_model.parameters())
total_params_trainable = sum(p.numel() for p in mtl_model.parameters() if p.requires_grad)
print("Number of parameters: ", total_params)
print("Number of trainable parameters: ", total_params_trainable)

total_params_llm = sum(p.numel() for p in llm.parameters())
total_params_trainable_llm = sum(p.numel() for p in llm.parameters() if p.requires_grad)
print("Number of parameters: ", total_params_llm)
print("Number of trainable parameters: ", total_params_trainable_llm)
print(total_params - total_params_llm)

Number of parameters:  141353926
Number of trainable parameters:  141353926
Number of parameters:  141304320
Number of trainable parameters:  141304320
49606


In [41]:
def train(mtl_model, optimizer, train_combined_loader, eta=1.0, num_epochs=5):

    mtl_model.train()

    # Record losses per epoch
    loss_per_epoch = []
    reddit_loss_per_epoch = []
    ucc_loss_per_epoch = []

    # Record accuracies per epoch
    total_accs = []
    reddit_accs = []
    ucc_accs = []

    for i in tqdm(range(num_epochs)):

        # Keep track of correct predictions for accuracy reporting.
        ucc_correct = 0
        reddit_correct = 0
        total_correct = 0
        num_preds = 0

        loss_per_batch = []
        reddit_losses = []
        ucc_losses = []

        for batch, batch_idx, dataloader_idx in train_combined_loader:

            # Get data batches for both tasks.
            ucc_batch = batch['a']
            reddit_batch = batch['b']

            # Push batches to device.
            ucc_inputs = {k: v.to(device) for k, v in ucc_batch.items() if k != 'labels'}
            reddit_inputs = {k: v.to(device) for k, v in reddit_batch.items() if k != 'labels'}
            ucc_labels = ucc_batch['labels'].to(device)
            reddit_labels = reddit_batch['labels'].to(device)

            # Get predictions for both tasks for the current batch of data.
            ucc_preds = mtl_model(**ucc_inputs, taskid=1)
            reddit_preds = mtl_model(**reddit_inputs, taskid=2)
            ucc_preds = ucc_preds.squeeze(dim=1)                # output is [Batch_size, 1] but we want [Batch_size]
            reddit_preds = reddit_preds.squeeze(dim=1)

            # Calculate loss.
            task1_loss = ucc_loss(ucc_preds, ucc_labels)
            task2_loss = reddit_loss(reddit_preds, reddit_labels)
            if eta == 1.0:
                loss = task1_loss + task2_loss
            else:
                loss = eta*task1_loss + (1-eta)*task2_loss

            # Record loss per batch
            loss_per_batch.append(loss.item())
            ucc_losses.append(task1_loss.item())
            reddit_losses.append(task2_loss.item())

            # Record accuracy for the current batch.
            num_preds += ucc_labels.shape[0]

            ucc_preds = torch.sigmoid(ucc_preds)                    # Convert logits to probabilities for multi-label classification
            ucc_preds_labels = (ucc_preds >= 0.5).long()            # Binarize predictions to 0 and 1
            ucc_correct += (ucc_preds_labels == ucc_labels).sum()   # Count up number of correct predictions for this batch

            reddit_preds = torch.sigmoid(reddit_preds)              # Repeat for reddit task
            reddit_preds_labels = (reddit_preds >= 0.5).long()
            reddit_correct += (reddit_preds_labels == reddit_labels).sum()

            # Update the model
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lr_scheduler.step()

            # Release memory on GPU
            del ucc_inputs, reddit_inputs, ucc_labels, reddit_labels
            del ucc_preds, ucc_preds_labels, reddit_preds, reddit_preds_labels, loss
            torch.cuda.empty_cache()

        # Record loss per epoch
        loss_per_epoch.append(loss_per_batch)
        reddit_loss_per_epoch.append(reddit_losses)
        ucc_loss_per_epoch.append(ucc_losses)

        # Calculate and record accuracies.
        ucc_accuracy = ucc_correct / (num_preds*5)
        ucc_accs.append(ucc_accuracy)

        reddit_accuracy = reddit_correct / num_preds
        reddit_accs.append(reddit_accuracy)

        total_accuracy = (ucc_correct + reddit_correct) / (num_preds + (5*num_preds))
        total_accs.append(total_accuracy)

    return loss_per_epoch, reddit_loss_per_epoch, ucc_loss_per_epoch, total_accs, reddit_accs, ucc_accs


In [42]:
# Run Training on model
loss_per_epoch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs = train(mtl_model, optimizer, train_combined_loader, eta=1.0, num_epochs=5)

# Save the model
state_dict = mtl_model.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_confident.tar'
torch.save(state_dict, model_name)

llm_state_dict = mtl_model.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_confident.tar'
torch.save(llm_state_dict, llm_name)

# Save training results
results_dict = {
    'loss_per_epoch': loss_per_epoch, 'reddit_losses': reddit_losses, 'ucc_losses': ucc_losses,
    'total_accs': total_accs, 'reddit_accs': reddit_accs, 'ucc_accs': ucc_accs
}
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_confident.pickle'
with open(results_file, 'wb') as handle:
    pickle.dump(results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


  0%|          | 0/5 [00:00<?, ?it/s]

In [43]:
# Load models for Evaluation

llm_state_dict = torch.load(llm_name)
saved_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
saved_llm.load_state_dict(llm_state_dict)

state_dict = torch.load(model_name)
saved_mtl_model = MTLTextClassification(llm=saved_llm)
saved_mtl_model.load_state_dict(state_dict)
saved_mtl_model = saved_mtl_model.to(device)

# Verify that the parameters are the same
#print("Original model\n", mtl_model.state_dict()['llm.embeddings.LayerNorm.weight'][0:5])
#print("Loaded model\n", saved_mtl_model.state_dict()['llm.embeddings.LayerNorm.weight'][0:5])
#print("\nOriginal model\n", mtl_model.state_dict()['ucc_out.bias'])
#print("Loaded model\n", saved_mtl_model.state_dict()['ucc_out.bias'])

# Create data loader for testing
test_combined_loader = CombinedLoader(test_loaders, 'max_size_cycle')

In [44]:
def test(mtl_model, train_combined_loader, eta=1.0):
    mtl_model.eval()

    # Record testing losses per batch
    loss_per_batch = []
    reddit_losses = []
    ucc_losses = []

    # Record accuracies per batch
    total_accs = []
    reddit_accs = []
    ucc_accs = []

    # Keep track of correct predictions for accuracy reporting.
    ucc_correct = 0
    reddit_correct = 0
    total_correct = 0
    num_preds = 0

    with torch.no_grad():
        for batch, batch_idx, dataloader_idx in test_combined_loader:

            # Get data batches for both tasks.
            ucc_batch = batch['a']
            reddit_batch = batch['b']

            # Push batches to device.
            ucc_inputs = {k: v.to(device) for k, v in ucc_batch.items() if k != 'labels'}
            reddit_inputs = {k: v.to(device) for k, v in reddit_batch.items() if k != 'labels'}
            ucc_labels = ucc_batch['labels'].to(device)
            reddit_labels = reddit_batch['labels'].to(device)

            # Get predictions for both tasks for the current batch of data.
            ucc_preds = mtl_model(**ucc_inputs, taskid=1)
            reddit_preds = mtl_model(**reddit_inputs, taskid=2)
            ucc_preds = ucc_preds.squeeze(dim=1)                # output is [Batch_size, 1] but we want [Batch_size]
            reddit_preds = reddit_preds.squeeze(dim=1)

            # Calculate loss.
            task1_loss = ucc_loss(ucc_preds, ucc_labels)
            task2_loss = reddit_loss(reddit_preds, reddit_labels)
            if eta == 1.0:
                loss = task1_loss + task2_loss
            else:
                loss = eta*task1_loss + (1-eta)*task2_loss

            loss_per_batch.append(loss.item())
            ucc_losses.append(task1_loss.item())
            reddit_losses.append(task2_loss.item())

            # Record accuracy for the current batch.
            num_preds += ucc_labels.shape[0]

            ucc_preds = torch.sigmoid(ucc_preds)                    # Convert logits to probabilities for multi-label classification
            ucc_preds_labels = (ucc_preds >= 0.5).long()            # Binarize predictions to 0 and 1
            ucc_correct += (ucc_preds_labels == ucc_labels).sum()   # Count up number of correct predictions for this batch

            reddit_preds = torch.sigmoid(reddit_preds)              # Repeat for reddit task
            reddit_preds_labels = (reddit_preds >= 0.5).long()
            reddit_correct += (reddit_preds_labels == reddit_labels).sum()

            # Release memory on GPU
            del ucc_inputs, reddit_inputs, ucc_labels, reddit_labels
            del ucc_preds, ucc_preds_labels, reddit_preds, reddit_preds_labels, loss
            torch.cuda.empty_cache()

    # Calculate and record accuracies.
    ucc_accuracy = ucc_correct / (num_preds * 5)
    ucc_accs.append(ucc_accuracy)

    reddit_accuracy = reddit_correct / num_preds
    reddit_accs.append(reddit_accuracy)

    total_accuracy = (ucc_correct + reddit_correct) / (num_preds + (5*num_preds))
    total_accs.append(total_accuracy)

    return loss_per_batch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs

In [45]:
loss_per_batch, reddit_losses_test, ucc_losses_test, total_accs_test, reddit_accs_test, ucc_accs_test = test(saved_mtl_model, test_combined_loader, eta=1.0)

# Save testing results
test_results_dict = {
    'loss_per_batch': loss_per_batch, 'reddit_losses': reddit_losses_test, 'ucc_losses': ucc_losses_test,
    'total_accs': total_accs_test, 'reddit_accs': reddit_accs_test, 'ucc_accs': ucc_accs_test
}
test_results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_confident.pickle'
with open(test_results_file, 'wb') as handle:
    pickle.dump(test_results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [36]:
# Clean up
del mtl_model, llm
torch.cuda.empty_cache()

In [ ]:
# Clean up
del saved_mtl_model, saved_llm
torch.cuda.empty_cache()

# Single-Task Learning Baselines



In [13]:
def stl_training(model, loader, optimizer, loss_fn, taskid):
    model.train()
    losses = []
    accs = []
    for epoch in tqdm(range(5)):
        correct_preds = 0.0
        num_preds = 0.0
        batch_losses = []
        for batch in loader:
            batch_labels = batch['labels']
            batch_labels = batch_labels.to(device)
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

            outputs = model(**batch, taskid=taskid)
            outputs = outputs.squeeze(dim=1)
            loss = loss_fn(outputs, batch_labels)
            batch_losses.append(loss.item())

            # Record accuracy for the current batch.
            num_preds += batch_labels.shape[0]
            preds = torch.sigmoid(outputs)                      # Convert logits to probabilities for multi-label classification
            preds_labels = (preds >= 0.5).long()                # Binarize predictions to 0 and 1
            correct_preds += (preds_labels == batch_labels).sum()   # Count up number of correct predictions for this batch

            # Update model
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lr_scheduler.step()

            # Release memory on GPU
            del batch, batch_labels
            del outputs, preds, preds_labels, loss
            torch.cuda.empty_cache()

        losses.append(batch_losses)

        if taskid == 1:
            epoch_acc = correct_preds / (5*num_preds)
        else:
            epoch_acc = correct_preds / (num_preds)
        accs.append(epoch_acc)

    return {'Loss': losses, 'Accuracy': accs}

In [14]:
def stl_testing(model, loader, loss_fn, taskid):
    model.eval()
    losses = []
    accs = []
    with torch.no_grad():
        correct_preds = 0.0
        num_preds = 0.0
        batch_losses = []
        for batch in loader:
            batch_labels = batch['labels']
            batch_labels = batch_labels.to(device)
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

            outputs = model(**batch, taskid=taskid)
            outputs = outputs.squeeze(dim=1)
            loss = loss_fn(outputs, batch_labels)
            batch_losses.append(loss.item())

            # Record accuracy for the current batch.
            num_preds += batch_labels.shape[0]
            preds = torch.sigmoid(outputs)                      # Convert logits to probabilities for multi-label classification
            preds_labels = (preds >= 0.5).long()                # Binarize predictions to 0 and 1
            correct_preds += (preds_labels == batch_labels).sum()   # Count up number of correct predictions for this batch

            # Release memory on GPU
            del batch, batch_labels
            del outputs, preds, preds_labels, loss
            torch.cuda.empty_cache()

        losses.append(batch_losses)

        if taskid == 1:
            acc = correct_preds / (5*num_preds)
        else:
            acc = correct_preds / (num_preds)

    return {'Loss': losses, 'Accuracy': acc}

## UCC STL Baseline

In [18]:
# Train model on UCC dataset using STL only
ucc_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
ucc_stl = MTLTextClassification(llm=ucc_llm)
ucc_stl = ucc_stl.to(device)
ucc_optimizer = torch.optim.AdamW(ucc_stl.parameters(), lr=5e-5)
ucc_train_loader = DataLoader(ucc_train_dataset, batch_size=4)
ucc_test_loader = DataLoader(ucc_eval_dataset, batch_size=4)
ucc_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=ucc_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the UCC STL Model and save the results
train_results = stl_training(ucc_stl, ucc_train_loader, ucc_optimizer, loss_fn=ucc_loss, taskid=1)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_train_results.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(train_results, f)

# Evaluate the UCC STL Model and save the results
test_results = stl_testing(ucc_stl, ucc_test_loader, loss_fn=ucc_loss, taskid=1)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_test_results.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(test_results, f)

  0%|          | 0/5 [00:00<?, ?it/s]

In [19]:
# Examine number of parameters
total_params = sum(p.numel() for p in ucc_stl.parameters())
total_params_trainable = sum(p.numel() for p in ucc_stl.parameters() if p.requires_grad)
print("Number of parameters: ", total_params)
print("Number of trainable parameters: ", total_params_trainable)

Number of parameters:  141353926
Number of trainable parameters:  141353926


In [21]:
del ucc_llm, ucc_stl, ucc_optimizer, ucc_train_loader, ucc_test_loader, ucc_loss, lr_scheduler, train_results, test_results
torch.cuda.empty_cache()

## Reddit STL Baseline

In [15]:
# Train model on UCC dataset using STL only
reddit_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
reddit_stl = MTLTextClassification(llm=reddit_llm)
reddit_stl = reddit_stl.to(device)
reddit_optimizer = torch.optim.AdamW(reddit_stl.parameters(), lr=5e-5)
reddit_train_loader = DataLoader(reddit_train_dataset, batch_size=4)
reddit_test_loader = DataLoader(reddit_eval_dataset, batch_size=4)
reddit_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=reddit_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the UCC STL Model and save the results
train_results = stl_training(reddit_stl, reddit_train_loader, reddit_optimizer, loss_fn=reddit_loss, taskid=2)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_train_results.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(train_results, f)

# Evaluate the UCC STL Model and save the results
test_results = stl_testing(reddit_stl, reddit_test_loader, loss_fn=reddit_loss, taskid=2)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_test_results.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(test_results, f)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [17]:
# Examine number of parameters
total_params = sum(p.numel() for p in reddit_stl.parameters())
total_params_trainable = sum(p.numel() for p in reddit_stl.parameters() if p.requires_grad)
print("Number of parameters: ", total_params)
print("Number of trainable parameters: ", total_params_trainable)

Number of parameters:  141353926
Number of trainable parameters:  141353926


# Hyperparameter search

In [ ]:
# Try different configs for LLM (dropout rate)

# Test different epochs, learning rates, and batch sizes

# Test different architectures for task heads

In [ ]:
# Clean up
del mtl_model, llm
torch.cuda.empty_cache()

## Task weighting: eta = 0.6
60% of ucc_loss + 40% of abuse_loss

In [ ]:
# Set up the LLM
llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
llm = llm.to(device)

# Define params of LLM as trainable
for param in llm.parameters():
    param.requires_grad = True

# Initialize Custom MTL Model and optimizer
mtl_model = MTLTextClassification(llm=llm)
mtl_model = mtl_model.to(device)
optimizer = torch.optim.AdamW(mtl_model.parameters(), lr=5e-5)

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
# Run Training on model
loss_per_epoch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs = train(mtl_model, optimizer, train_combined_loader, eta=0.6, num_epochs=5)

# Save the model
state_dict = mtl_model.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_eta_60-40'
torch.save(state_dict, model_name)

llm_state_dict = mtl_model.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_eta_60-40.tar'
torch.save(llm_state_dict, llm_name)

# Save training results
results_dict = {
    'loss_per_epoch': loss_per_epoch, 'reddit_losses': reddit_losses, 'ucc_losses': ucc_losses,
    'total_accs': total_accs, 'reddit_accs': reddit_accs, 'ucc_accs': ucc_accs
}
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_eta_60_40.pickle'
with open(results_file, 'wb') as handle:
    pickle.dump(results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Load models for Evaluation
llm_state_dict = torch.load(llm_name)
saved_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
saved_llm.load_state_dict(llm_state_dict)

state_dict = torch.load(model_name)
saved_mtl_model = MTLTextClassification(llm=saved_llm)
saved_mtl_model.load_state_dict(state_dict)
saved_mtl_model = saved_mtl_model.to(device)


In [ ]:
# Run evaluation
loss_per_batch, reddit_losses_test, ucc_losses_test, total_accs_test, reddit_accs_test, ucc_accs_test = test(saved_mtl_model, test_combined_loader, eta=0.6)

# Save testing results
test_results_dict = {
    'loss_per_batch': loss_per_batch, 'reddit_losses': reddit_losses_test, 'ucc_losses': ucc_losses_test,
    'total_accs': total_accs_test, 'reddit_accs': reddit_accs_test, 'ucc_accs': ucc_accs_test
}
test_results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_eta_60_40.pickle'
with open(test_results_file, 'wb') as handle:
    pickle.dump(test_results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open(results_file, 'rb') as f:
    res = pickle.load(f)

print(res.keys())
print(res['ucc_accs'])

dict_keys(['loss_per_epoch', 'reddit_losses', 'ucc_losses', 'total_accs', 'reddit_accs', 'ucc_accs'])
[tensor(0.5907, device='cuda:0'), tensor(0.5969, device='cuda:0'), tensor(0.6240, device='cuda:0'), tensor(0.6701, device='cuda:0'), tensor(0.7040, device='cuda:0')]


## Task Weighting


In [ ]:
# Clean up
del mtl_model, llm
del optimizer
torch.cuda.empty_cache()

In [ ]:
# Set up the LLM
llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
llm = llm.to(device)

# Define params of LLM as trainable
for param in llm.parameters():
    param.requires_grad = True

# Initialize Custom MTL Model and optimizer
mtl_model = MTLTextClassification(llm=llm)
mtl_model = mtl_model.to(device)
optimizer = torch.optim.AdamW(mtl_model.parameters(), lr=5e-5)

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
# Run Training on model
loss_per_epoch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs = train(mtl_model, optimizer, train_combined_loader, eta=0.4, num_epochs=5)

# Save the model
state_dict = mtl_model.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_eta_40-60'
torch.save(state_dict, model_name)

llm_state_dict = mtl_model.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_eta_40-60.tar'
torch.save(llm_state_dict, llm_name)

# Save training results
results_dict = {
    'loss_per_epoch': loss_per_epoch, 'reddit_losses': reddit_losses, 'ucc_losses': ucc_losses,
    'total_accs': total_accs, 'reddit_accs': reddit_accs, 'ucc_accs': ucc_accs
}
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_eta_40_60.pickle'
with open(results_file, 'wb') as handle:
    pickle.dump(results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Load models for Evaluation
llm_state_dict = torch.load(llm_name)
saved_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
saved_llm.load_state_dict(llm_state_dict)

state_dict = torch.load(model_name)
saved_mtl_model = MTLTextClassification(llm=saved_llm)
saved_mtl_model.load_state_dict(state_dict)
saved_mtl_model = saved_mtl_model.to(device)

In [ ]:
# Run evaluation
loss_per_batch, reddit_losses_test, ucc_losses_test, total_accs_test, reddit_accs_test, ucc_accs_test = test(saved_mtl_model, test_combined_loader, eta=0.4)

# Save testing results
test_results_dict = {
    'loss_per_batch': loss_per_batch, 'reddit_losses': reddit_losses_test, 'ucc_losses': ucc_losses_test,
    'total_accs': total_accs_test, 'reddit_accs': reddit_accs_test, 'ucc_accs': ucc_accs_test
}
test_results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_eta_40_60.pickle'
with open(test_results_file, 'wb') as handle:
    pickle.dump(test_results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

Typically, as mentioned by Devlin et al. [1], for a classification task, we use the first output vector of a sentence as input for the rest of the classification model, since this first vector “encodes” information about the overall sentence. Alternatively a pooling average of all output vectors could also be used as input for the classifier.